In [1]:
# libraries we needed to installed
!pip install -qU langchain langchain-community langchain-core llama-cpp-python chromadb sentence-transformers pypdf unstructured[all-docs]
!pip install -q transformers==4.44.2 einops torch torchvision pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 14.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 77.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 63.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.1 MB/s eta 0:00:00
  

In [4]:

import os
import torch
from PIL import Image
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM , AutoTokenizer

# Part 1 of the task : Rag pipline
print("\n[PART 1: RAG Pipeline")

# 1.1 Igent & Chunk
pdf_path="sample_document.pdf"
if not os.path.exists(pdf_path):
    os.system('wget -q -O sample_document.pdf "https://arxiv.org/pdf/1706.03762.pdf"')

print(f"Loading document: {pdf_path}")
loader = PyPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)
print(f"Document split into {len(chunks)} chunks.")

# 1.2 Embed & Index (ChromaDB)
print("Initializing Local Embeddings (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="./chroma_db")

# 1.3 Local LLM Setup (Mistral 7B)
model_path = "./mistral.gguf"
if not os.path.exists(model_path):
    print("Downloading Mistral-7B model...")
    os.system('wget -q -O mistral.gguf https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf')

print("Loading Mistral-7B LLM into Memory...")
llm = LlamaCpp(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=2048,
    temperature=0.1,
    verbose=False
)

# 1.4 Retrieval & QA Generation
query = "Based on the document, what is the role of the attention mechanism?"
print(f"\nUser Query: {query}")

# Explicit Retrieval for absolute stability
retrieved_docs = vector_db.similarity_search(query, k=2)
context_text = "\n---\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""You are a helpful and precise AI assistant.
Use ONLY the following context to answer the question. If you don't know, say "I don't know".

Context:
{context_text}

Question: {query}

Answer:"""

answer = llm.invoke(prompt)
source_page = retrieved_docs[0].metadata.get('page', 'Unknown')

print(f"System Answer: {answer.strip()}")
print(f"Citation: Extracted from Page {source_page}")


# Part 2 : Vision Pipline
print("\n" + "="*50)
print("\n[PART 2: Vision Pipeline")


# 2.1 Load Image
img_path = "sample_invoice.png"
if not os.path.exists(img_path):
    os.system('wget -q -O sample_invoice.png "https://templates.invoicehome.com/invoice-template-us-neat-750px.png"')

image = Image.open(img_path)
print("Invoice Image Loaded.")

# 2.2 Local VLM Setup (Moondream2)
print("Loading Moondream2 Vision Model...")
vlm_model_id = "vikhyatk/moondream2"
revision = "2024-08-26"

tokenizer = AutoTokenizer.from_pretrained(vlm_model_id, revision=revision)
vlm_model = AutoModelForCausalLM.from_pretrained(
    vlm_model_id, trust_remote_code=True, revision=revision, torch_dtype=torch.float16
).to("cuda")

# 2.3 Visual Extraction
vision_prompt = "What is the total amount due on this invoice? Please read the numbers carefully."
print(f"\nVision Task: {vision_prompt}")

enc_image = vlm_model.encode_image(image)
vision_answer = vlm_model.answer_question(enc_image, vision_prompt, tokenizer)

print(f"Vision AI Extracted Value: {vision_answer.strip()}")




[PART 1: RAG Pipeline
Loading document: sample_document.pdf
Document split into 52 chunks.
Initializing Local Embeddings (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading Mistral-7B LLM into Memory...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



User Query: Based on the document, what is the role of the attention mechanism?
System Answer: The attention mechanism in the context of the document is a function that maps a query and a set of key-value pairs to an output. This output is computed as a weighted sum of the values, where the weights are determined by the similarity between the query and each key. The role of the attention mechanism is to selectively focus on the most relevant parts of the input data when processing a query.
Citation: Extracted from Page 2


[PART 2: Vision Pipeline
Invoice Image Loaded.
Loading Moondream2 Vision Model...


...

fourier_features.py:   0%|          | 0.00/558 [00:00<?, ?B/s]

vision_encoder.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.74G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


Vision Task: What is the total amount due on this invoice? Please read the numbers carefully.
Vision AI Extracted Value: $154.06
